In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.model_selection import KFold, train_test_split
from torch.utils.data import TensorDataset, DataLoader, Subset
import numpy as np
import prep as dtpr

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

dataset = dtpr.prep_data(True)
print(dataset.head())


# dataset
X_n = dataset.iloc[:, :-1].values.astype(np.float32)
y_n = dataset.iloc[:, -1].values.astype(np.int64)


# Hyperparameters
num_classes = len(np.unique(y_n))
input_dim = X_n.shape[1]
train_size = int(X_n.shape[0]*0.75)
hidden_dim = 64
epochs = 10
batch_size = 128
lr = 0.001
k_folds = 5
bins = 20
lambda_dist = 1.0  # Weight for distribution matching loss


# Split into train+val and test
X = torch.from_numpy(X_n) 
y = torch.from_numpy(y_n)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

trainval_dataset = TensorDataset(X_trainval, y_trainval)
test_dataset = TensorDataset(X_test, y_test)

# Model definition
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, temperature=1.0):
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)
        probs = F.softmax(logits / temperature, dim=1)
        return logits, probs

    Area  Perimeter  MajorAxisLength  MinorAxisLength  AspectRation  \
0  28395    610.291       208.178117       173.888747      1.197191   
1  28734    638.018       200.524796       182.734419      1.097356   
2  29380    624.110       212.826130       175.931143      1.209713   
3  30008    645.884       210.557999       182.516516      1.153638   
4  30140    620.134       201.847882       190.279279      1.060798   

   Eccentricity  ConvexArea  EquivDiameter    Extent  Solidity  roundness  \
0      0.549812       28715     190.141097  0.763923  0.988856   0.958027   
1      0.411785       29172     191.272750  0.783968  0.984986   0.887034   
2      0.562727       29690     193.410904  0.778113  0.989559   0.947849   
3      0.498616       30724     195.467062  0.782681  0.976696   0.903936   
4      0.333680       30417     195.896503  0.773098  0.990893   0.984877   

   Compactness  ShapeFactor1  ShapeFactor2  ShapeFactor3  ShapeFactor4  Class  
0     0.913358      0.007332  

/home/local1/miniconda3/envs/amd/lib/python3.10/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0
/home/local1/workspace/danuka/Quantification/data/dry_beans/prep.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dta.Class = dta.Class.replace({'BARBUNYA': 0, 'BOMBAY': 1, 'CALI': 2,


## Test Helinger distance Function

In [2]:
# Hellinger distance
def hellinger_distance(p, q):
    return torch.sqrt(0.5 * torch.sum((torch.sqrt(p) - torch.sqrt(q)) ** 2))

In [3]:
# Unit test
# create two random histograms

# histogram 1 ----------------------------------------------------------------------------------------------
# Generate from a normal distribution (mean=0, std=1)
tensor = torch.randn(100)

# Optionally normalize to [0, 1]
tensor_min = tensor.min()
tensor_max = tensor.max()
hist1 = (tensor - tensor_min) / (tensor_max - tensor_min)

# histogram 2 ----------------------------------------------------------------------------------------------
# Generate from a normal distribution (mean=0, std=1)
tensor = torch.randn(100)

# Optionally normalize to [0, 1]
tensor_min = tensor.min()
tensor_max = tensor.max()
hist2 = (tensor - tensor_min) / (tensor_max - tensor_min)

# get hellinger distance
hd = hellinger_distance(hist1, hist2)

print(hist1)
print(hist2)
print('hd: ', hd)


tensor([0.9342, 0.8348, 0.7023, 0.0228, 0.6520, 0.2197, 0.4889, 0.1360, 0.3287,
        0.8713, 0.4100, 0.1814, 0.3342, 0.3722, 0.3249, 0.6710, 0.8699, 0.4626,
        0.3863, 0.5980, 0.3273, 0.7424, 0.6797, 0.8785, 0.7878, 0.7917, 0.6367,
        0.8004, 0.4463, 0.5081, 0.4418, 0.6930, 0.1857, 0.3018, 0.4482, 0.8868,
        0.5708, 0.4027, 0.5678, 0.3236, 0.1466, 0.7237, 0.2998, 0.3628, 0.2107,
        0.9785, 0.2196, 0.3884, 0.2921, 0.3499, 0.5163, 0.6175, 0.3884, 0.7679,
        0.3147, 0.3323, 0.1815, 0.5068, 0.4843, 0.6514, 0.4766, 0.9156, 0.2310,
        0.8114, 0.8253, 0.6922, 1.0000, 0.6169, 0.5770, 0.4541, 0.2603, 0.7875,
        0.4598, 0.6171, 0.5115, 0.5950, 0.6286, 0.3536, 0.0000, 0.3290, 0.5011,
        0.4221, 0.1957, 0.3664, 0.6435, 0.6305, 0.7398, 0.3969, 0.3453, 0.6285,
        0.5411, 0.4179, 0.5586, 0.7864, 0.4984, 0.4301, 0.2757, 0.5265, 0.5778,
        0.6382])
tensor([0.7115, 0.5705, 0.4640, 0.6675, 0.7791, 0.3939, 0.7751, 0.4769, 0.7933,
        0.6257, 0.7101,

## Test Distribution Matching

In [ ]:
# Distribution matching loss
def distribution_matching_loss(val_histograms, train_histograms, confusion_weights, num_classes):
    distances = []
    for pred in range(num_classes):
        combined_train_hist = torch.zeros_like(next(iter(train_histograms.values())))
        for actual in range(num_classes):
            weight = confusion_weights[actual, pred]
            combined_train_hist += weight * train_histograms[(actual, pred)]

        # Normalize combined_train_hist so it becomes a proper histogram
        combined_train_hist /= (combined_train_hist.sum() + 1e-8)

        # Normalize validation histogram for fair distance computation
        val_hist = val_histograms[pred] / (val_histograms[pred].sum() + 1e-8)

        dist = hellinger_distance(val_hist, combined_train_hist)
        distances.append(dist)

    return torch.max(torch.stack(distances))

In [ ]:
# UNIT TEST
nc = 3

histograms = {(pred, actual): torch.zeros(bins, device=device)
                  for pred in range(num_classes)
                  for actual in range(num_classes)}

# generate dummy dataset
for a in range(nc):
    for p in range(nc):
        # Generate from a normal distribution (mean=0, std=1)
        tensor = torch.randn(100)
        # Optionally normalize to [0, 1]
        tensor_min = tensor.min()
        tensor_max = tensor.max()
        hist2 = (tensor - tensor_min) / (tensor_max - tensor_min)

        histograms[(p, a)]




In [ ]:
def collect_train_histograms(model, dataloader, num_classes, bins):
    model.eval()
    histograms = {(pred, actual): torch.zeros(bins, device=device)
                  for pred in range(num_classes)
                  for actual in range(num_classes)}

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, probs = model(inputs)

            for i in range(probs.size(0)):
                prob_vector = probs[i]
                pred_class = torch.argmax(prob_vector).item()
                actual_class = labels[i].item()

                prob_value = prob_vector[pred_class].item()  # or prob_vector[actual_class].item() depending on what you want
                bin_idx = min(int(prob_value * bins), bins - 1)
                histograms[(pred_class, actual_class)][bin_idx] += 1

    # Normalize histograms
    for key in histograms:
        histograms[key] /= (histograms[key].sum() + 1e-8)

    return histograms

In [ ]:
def collect_val_histograms(probs, num_classes, bins):
    histograms = {pred: torch.zeros(bins, device=probs.device) for pred in range(num_classes)}
    for i in range(probs.size(0)):
        prob_vector = probs[i]
        pred_class = torch.argmax(prob_vector).item()
        prob_value = prob_vector[pred_class].item()
        bin_idx = min(int(prob_value * bins), bins - 1)
        histograms[pred_class][bin_idx] += 1
    return histograms  # <- no normalization

In [ ]:

def compute_confusion_weights(model, dataloader, num_classes):
    model.eval()
    conf_matrix = torch.zeros((num_classes, num_classes), device=device)
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, probs = model(inputs)
            preds = torch.argmax(probs, dim=1)
            for i in range(labels.size(0)):
                actual = labels[i].item()
                predicted = preds[i].item()
                conf_matrix[actual, predicted] += 1

    return conf_matrix  # <- raw integer counts

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.model_selection import KFold, train_test_split
from torch.utils.data import TensorDataset, DataLoader, Subset
import numpy as np
import prep as dtpr

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

dataset = dtpr.prep_data(True)
print(dataset.head())


# dataset
X_n = dataset.iloc[:, :-1].values.astype(np.float32)
y_n = dataset.iloc[:, -1].values.astype(np.int64)


# Hyperparameters
num_classes = len(np.unique(y_n))
input_dim = X_n.shape[1]
train_size = int(X_n.shape[0]*0.75)
hidden_dim = 64
epochs = 1
batch_size = 128
lr = 0.001
k_folds = 5
bins = 100
lambda_dist = 1.0  # Weight for distribution matching loss


# Split into train+val and test
X = torch.from_numpy(X_n) 
y = torch.from_numpy(y_n)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

trainval_dataset = TensorDataset(X_trainval, y_trainval)
test_dataset = TensorDataset(X_test, y_test)

# Model definition
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, temperature=1.0):
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)
        probs = F.softmax(logits / temperature, dim=1)
        return logits, probs
    
# Loss weight learning
class LossWeightLearner(nn.Module):
    def __init__(self):
        super(LossWeightLearner, self).__init__()
        self.log_sigma1 = nn.Parameter(torch.zeros(1))  # for loss1
        self.log_sigma2 = nn.Parameter(torch.zeros(1))  # for loss2

    def forward(self, loss1, loss2):
        weighted_loss = (torch.exp(-self.log_sigma1) * loss1 + self.log_sigma1) + \
                        (torch.exp(-self.log_sigma2) * loss2 + self.log_sigma2)
        return weighted_loss

# Full model wrapper
class FullModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(FullModel, self).__init__()
        self.mlp = SimpleMLP(input_dim, hidden_dim, num_classes)
        self.loss_weights = LossWeightLearner()

    def forward(self, x, temperature=1.0):
        return self.mlp(x, temperature)
    


# Hellinger distance
def hellinger_distance(p, q):
    return torch.sqrt(0.5 * torch.sum((torch.sqrt(p) - torch.sqrt(q)) ** 2))

# Distribution matching loss
def distribution_matching_loss(val_histograms, train_histograms, confusion_weights, num_classes):
    distances = []
    for pred in range(num_classes):
        combined_train_hist = torch.zeros_like(next(iter(train_histograms.values())))
        for actual in range(num_classes):
            weight = confusion_weights[actual, pred]
            combined_train_hist += weight * train_histograms[(actual, pred)]

        # Normalize combined_train_hist so it becomes a proper histogram
        combined_train_hist /= (combined_train_hist.sum() + 1e-8)

        # Normalize validation histogram for fair distance computation
        val_hist = val_histograms[pred] / (val_histograms[pred].sum() + 1e-8)

        dist = hellinger_distance(val_hist, combined_train_hist)
        distances.append(dist)

    return torch.max(torch.stack(distances))

def collect_train_histograms(model, dataloader, num_classes, bins):
    model.eval()
    histograms = {(pred, actual): torch.zeros(bins, device=device)
                  for pred in range(num_classes)
                  for actual in range(num_classes)}

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, probs = model(inputs)

            for i in range(probs.size(0)):
                prob_vector = probs[i]
                pred_class = torch.argmax(prob_vector).item()
                actual_class = labels[i].item()

                prob_value = prob_vector[pred_class].item()  # or prob_vector[actual_class].item() depending on what you want
                bin_idx = min(int(prob_value * bins), bins - 1)
                histograms[(pred_class, actual_class)][bin_idx] += 1

    # Normalize histograms
    for key in histograms:
        histograms[key] /= (histograms[key].sum() + 1e-8)

    return histograms

def collect_val_histograms(probs, num_classes, bins):
    histograms = {pred: torch.zeros(bins, device=probs.device) for pred in range(num_classes)}
    for i in range(probs.size(0)):
        prob_vector = probs[i]
        pred_class = torch.argmax(prob_vector).item()
        prob_value = prob_vector[pred_class].item()
        bin_idx = min(int(prob_value * bins), bins - 1)
        histograms[pred_class][bin_idx] += 1
    return histograms  # <- no normalization

def compute_confusion_weights(model, dataloader, num_classes):
    model.eval()
    conf_matrix = torch.zeros((num_classes, num_classes), device=device)
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, probs = model(inputs)
            preds = torch.argmax(probs, dim=1)
            for i in range(labels.size(0)):
                actual = labels[i].item()
                predicted = preds[i].item()
                conf_matrix[actual, predicted] += 1

    return conf_matrix  # <- raw integer counts



# K-Fold Cross-Validation
kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)

while(True):
    if(batch_size<train_size):
        for fold, (train_ids, val_ids) in enumerate(kfold.split(trainval_dataset)):
            print(f'FOLD {fold + 1}')
            
            train_subsampler = Subset(trainval_dataset, train_ids)
            val_subsampler = Subset(trainval_dataset, val_ids)

            train_loader = DataLoader(train_subsampler, batch_size=batch_size, shuffle=True)
            val_loader = DataLoader(val_subsampler, batch_size=batch_size)

            model = FullModel(input_dim, hidden_dim, num_classes).to(device)
            optimizer = optim.Adam(model.parameters(), lr=lr)
            criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

            # Training loop
            for epoch in range(epochs):
                model.train()
                total_loss = 0

                for inputs, labels in train_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    optimizer.zero_grad()

                    logits, probs = model(inputs)
                    ce_loss = criterion(logits, labels)

                    # Distribution matching loss on validation data (use small val batch for efficiency)
                    model.eval()
                    val_inputs, _ = next(iter(val_loader))
                    val_inputs = val_inputs.to(device)
                    with torch.no_grad():
                        val_logits, val_probs = model(val_inputs)
                    
                    train_histograms = collect_train_histograms(model, train_loader, num_classes, bins)
                    # print(train_histograms)
                    conf_matrix = compute_confusion_weights(model, train_loader, num_classes)
                    print(conf_matrix)

                    val_histograms = collect_val_histograms(val_probs, num_classes, bins)
                    dist_loss = distribution_matching_loss(val_histograms, train_histograms, conf_matrix, num_classes)

                    model.train()


                    # Combine losses using learned weights
                    total_loss = model.loss_weights(ce_loss, dist_loss)

                    loss = total_loss
                    loss.backward()
                    optimizer.step()

                    total_loss += loss.item()

                avg_loss = total_loss / len(train_loader)
                print(f'Epoch {epoch+1}/{epochs} | Loss: {avg_loss.item():.4f}')
        batch_size = batch_size*2
        
    else:
        break

# Final test set inference (optional)
model.eval()
test_loader = DataLoader(test_dataset, batch_size=batch_size)
all_probs, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        logits, probs = model(inputs)
        all_probs.append(probs.cpu())
        all_labels.append(labels)

all_probs = torch.cat(all_probs)
all_labels = torch.cat(all_labels)

print("Test Set Probabilities Shape:", all_probs.shape)
print("Test Set Labels Shape:", all_labels.shape)
print(all_probs[:5])
print(all_labels[:5])

# Save test set probabilities and labels for later use
# np.save("test_probs.npy", all_probs.numpy())
# np.save("test_labels.npy", all_labels.numpy())



# saving training set predictions and targets to get histograms
train_loader = DataLoader(trainval_dataset, batch_size=batch_size)
all_probs, all_labels = [], []

with torch.no_grad():
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        logits, probs = model(inputs)
        all_probs.append(probs.cpu())
        all_labels.append(labels)

all_probs = torch.cat(all_probs)
all_labels = torch.cat(all_labels)

# Save test set probabilities and labels for later use
# np.save("train_probs.npy", all_probs.numpy())
# np.save("train_labels.npy", all_labels.numpy())


/home/local1/workspace/danuka/Quantification/data/dry_beans/prep.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dta.Class = dta.Class.replace({'BARBUNYA': 0, 'BOMBAY': 1, 'CALI': 2,


    Area  Perimeter  MajorAxisLength  MinorAxisLength  AspectRation  \
0  28395    610.291       208.178117       173.888747      1.197191   
1  28734    638.018       200.524796       182.734419      1.097356   
2  29380    624.110       212.826130       175.931143      1.209713   
3  30008    645.884       210.557999       182.516516      1.153638   
4  30140    620.134       201.847882       190.279279      1.060798   

   Eccentricity  ConvexArea  EquivDiameter    Extent  Solidity  roundness  \
0      0.549812       28715     190.141097  0.763923  0.988856   0.958027   
1      0.411785       29172     191.272750  0.783968  0.984986   0.887034   
2      0.562727       29690     193.410904  0.778113  0.989559   0.947849   
3      0.498616       30724     195.467062  0.782681  0.976696   0.903936   
4      0.333680       30417     195.896503  0.773098  0.990893   0.984877   

   Compactness  ShapeFactor1  ShapeFactor2  ShapeFactor3  ShapeFactor4  Class  
0     0.913358      0.007332  

In [11]:
y.shape

torch.Size([13611])

In [13]:


print(len(trainval_dataset))             # prints 100
print(trainval_dataset.tensors[0].shape)  # prints torch.Size([100, 10])


print(len(test_dataset))             # prints 100
print(test_dataset.tensors[0].shape)  # prints torch.Size([100, 10])


10888
torch.Size([10888, 16])
2723
torch.Size([2723, 16])


In [7]:
all_probs[0]

tensor([0., 1., 0., 0., 0., 0., 0.])

In [8]:
idx = torch.argmax(all_probs, dim=1)

In [6]:
all_labels

tensor([5, 4, 2,  ..., 0, 4, 1])

In [9]:
idx

tensor([1, 1, 1,  ..., 1, 1, 1])

In [ ]:
import math

In [ ]:
def get_draw_size(y_cts, dt, train_distr, test_distr, C=None):
    if len(train_distr) != len(test_distr):
        raise ValueError("training and test distributions are not the same length")

    if C is None:
        C = sum(y_cts)

    constraints = [C] + [y_cts[i] / (dt[0] * train_distr[i] + dt[1] * test_distr[i])
                         for i in range(len(y_cts))]

    return np.floor(min(constraints))

In [ ]:
def synthetic_draw(n_y, n_classes, y_cts, y_idx, dt_distr, train_distr, test_distr, seed=4711):
    if len(train_distr) != len(test_distr):
        raise ValueError("training and test distributions are not the same length")

    if len(y_cts) != len(train_distr):
        raise ValueError("Length of training distribution does not match number of classes")

    if len(dt_distr) != 2:
        raise ValueError("Length of train/test-split has to equal 2")

    if not math.isclose(np.sum(dt_distr), 1.0):
        raise ValueError("Elements of train/test-split do not sum to 1")

    if not math.isclose(np.sum(train_distr), 1.0):
        raise ValueError("Elements of train distribution do not sum to 1")

    if not math.isclose(np.sum(test_distr), 1.0):
        raise ValueError("Elements of test distribution do not sum to 1")

    n = get_draw_size(y_cts, dt_distr, train_distr, test_distr, C=n_y)

    train_cts = (n * dt_distr[0] * train_distr).astype(int)
    if min(train_cts) == 0:
        raise ValueError("Under given input distributions a class would miss in training")

    test_cts = (n * dt_distr[1] * test_distr).astype(int)

    # fix seed for reproducibility
    np.random.seed(seed)

    train_list = [np.random.choice(y_idx[i], size=train_cts[i], replace=False) for i in range(n_classes)]
    y_idx = [np.setdiff1d(y_idx[i], train_list[i]) for i in range(n_classes)]
    test_list = [np.random.choice(y_idx[i], size=test_cts[i], replace=False) if np.size(y_idx[i]) > 0 else [] for i in
                 range(n_classes)]

    train_index = np.concatenate(train_list)
    test_index = np.concatenate(test_list).astype(int)

    n_train = train_index.shape[0]
    n_test = test_index.shape[0]
    M = n_train + n_test
    r_train = n_train * 1.0 / M
    r_test = n_test * 1.0 / M

    train_ratios = train_cts * 1.0 / n_train
    test_ratios = test_cts * 1.0 / n_test

    stats_vec = np.concatenate(
        [np.array([M, n_train, n_test, r_train, r_test]), train_cts, train_ratios, test_cts, test_ratios])

    return train_index, test_index, stats_vec

In [ ]:
seed = 42
n_classes = 7
N = 13611
y_cts = np.array([1322,  522, 1630, 3546, 1928, 2027, 2636])


In [ ]:
for dt_distr in dt_ratios:

    for train_distr in train_ds:

        for test_distr in test_ds:
            
            print('Training and Test dists:')
            print(dt_distr)
            print(train_distr)
            print(test_distr)

            train_index, test_index, stats_vec = synthetic_draw(N, n_classes, y_cts, y_idx, dt_distr, train_distr, test_distr, seed)
            
            X_train, y_train = X[train_index], y[train_index]
            X_test = X[test_index]

In [2]:
import numpy as np

In [30]:
a = np.array([[1,2],[5,6]])
b = a/(np.sum(a,axis=1)).reshape(-1, 1)

In [31]:
b

array([[0.33333333, 0.66666667],
       [0.45454545, 0.54545455]])

In [25]:
for i in range(2):
    for j in range(2):
        print(a[i][j],' ',b[i][j])

1   0.3333333333333333
2   0.18181818181818182
5   1.6666666666666667
6   0.5454545454545454


In [29]:
np.sum(a,axis=1).reshape(-1, 1)

array([[ 3],
       [11]])

In [35]:
4%3

1